# cd_A_weakfaith — 收敛时间 & Iter 数 vs d

固定 n=1000, degree=2，对 d∈{50,60,70,80} 测量 early-stop 收敛所需的 coordinate-descent 步数和时间。
收敛判据：连续 CONV_PATIENCE 次 epoch 检查，分数相对改善 < CONV_TOL。

In [ ]:
import os, sys, time, warnings
import concurrent.futures
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "calm_dataset.py").exists() and (p / "coordinate_descent").exists():
            return p
    raise RuntimeError(f"repo root not found from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from calm_dataset import CalmDataset
from coordinate_descent.cd_A_weakfaith import dag_coordinate_descent_l0_weakfaith
print(f"REPO_ROOT: {REPO_ROOT}")
print("cd_A_weakfaith: OK")

In [ ]:
ALGO_NAME   = "cd_A_weakfaith"
TAG         = "cd_A_convergence"

D_LIST      = [50, 60, 70, 80]
N_SAMPLES   = 1000
TRIALS      = 3
DEGREE      = 2.0
GRAPH_TYPE  = "ER"
SEED_BASE   = 42
TIMEOUT_SEC = 3600

T_MAX        = 10_000_000   # max coordinate steps (large, early-stop will trigger first)
CONV_TOL     = 1e-4         # relative score improvement threshold
CONV_PATIENCE = 5           # consecutive non-improving epoch checks before stopping

os.makedirs(str(REPO_ROOT / "experiments" / "results"), exist_ok=True)

In [ ]:
def run_with_result_and_timeout(fn, timeout):
    """Returns (result, elapsed_sec, error_str_or_None)."""
    holder = [None]
    def _wrap():
        holder[0] = fn()
    ex = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    fut = ex.submit(_wrap)
    t0 = time.perf_counter()
    try:
        fut.result(timeout=timeout)
        elapsed = time.perf_counter() - t0
        ex.shutdown(wait=False)
        return holder[0], elapsed, None
    except concurrent.futures.TimeoutError:
        ex.shutdown(wait=False)
        return None, float(timeout), "TIMEOUT"
    except Exception as e:
        elapsed = time.perf_counter() - t0
        ex.shutdown(wait=False)
        return None, elapsed, str(e)


def make_data(d, seed):
    ds = CalmDataset(
        n=N_SAMPLES, d=d, graph_type=GRAPH_TYPE,
        degree=DEGREE, sem_type="gauss",
        noise_ratio=4.0, seed=int(seed),
    )
    return ds.X


def run_algo(X, seed):
    """Returns (n_iters, final_score)."""
    n = X.shape[0]
    S = X.T @ X / n
    d = X.shape[1]
    check_every = d * (d + 1) // 2   # one "epoch"
    _, _, final_obj, history = dag_coordinate_descent_l0_weakfaith(
        S=S, T=T_MAX, seed=seed, threshold=0.05, lambda_l0=0.2,
        faithfulness_tau=0.05, screening=["corr", "pcorr"], combine="union",
        early_stop=True, check_every=check_every,
        tol=CONV_TOL, patience=CONV_PATIENCE,
        return_history=True,
    )
    return len(history), float(final_obj)

## 主循环

In [ ]:
records = []
rng = np.random.default_rng(SEED_BASE)
seeds = rng.integers(0, 10**9, size=(len(D_LIST), TRIALS))

for di, d in enumerate(D_LIST):
    for t in range(TRIALS):
        X = make_data(d, int(seeds[di, t]))
        result, elapsed, err = run_with_result_and_timeout(
            lambda X=X, s=int(seeds[di, t]): run_algo(X, s), TIMEOUT_SEC
        )
        if err == "TIMEOUT":
            print(f"d={d:3d}  trial={t+1}  TIMEOUT (>{TIMEOUT_SEC}s)")
            records.append({"d": d, "trial": t+1, "n_iters": np.nan,
                            "time_sec": elapsed, "final_score": np.nan, "converged": False})
        elif err:
            print(f"d={d:3d}  trial={t+1}  ERROR: {err}")
        else:
            n_iters, final_score = result
            print(f"d={d:3d}  trial={t+1}  n_iters={n_iters:>10,d}  "
                  f"time={elapsed:7.2f}s  score={final_score:.4f}")
            records.append({"d": d, "trial": t+1, "n_iters": n_iters,
                            "time_sec": elapsed, "final_score": final_score, "converged": True})

df = pd.DataFrame(records)
print()
print(df.to_string(index=False))

## 汇总 & 可视化

In [ ]:
df_conv = df[df["converged"]]
agg = df_conv.groupby("d").agg(
    mean_iters=("n_iters", "mean"),
    std_iters=("n_iters", "std"),
    mean_time=("time_sec", "mean"),
    std_time=("time_sec", "std"),
    mean_score=("final_score", "mean"),
).reset_index()
print(agg.to_string(index=False))

if len(agg) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"{ALGO_NAME} — convergence vs d  (n={N_SAMPLES}, ER degree={DEGREE})", fontsize=13)

    ax = axes[0]
    ax.errorbar(agg["d"], agg["mean_iters"], yerr=agg["std_iters"].fillna(0),
                fmt="o-", capsize=4)
    ax.set_xlabel("d"); ax.set_ylabel("iters to converge")
    ax.set_title("Iterations to convergence")
    ax.grid(True, ls="--", alpha=0.4)

    ax = axes[1]
    ax.errorbar(agg["d"], agg["mean_time"], yerr=agg["std_time"].fillna(0),
                fmt="o-", capsize=4, color="orange")
    ax.set_xlabel("d"); ax.set_ylabel("time (s)")
    ax.set_title("Time to convergence")
    ax.grid(True, ls="--", alpha=0.4)

    plt.tight_layout()
    out_png = REPO_ROOT / "experiments" / "results" / f"{TAG}.png"
    plt.savefig(out_png, dpi=120)
    plt.show()
    print(f"figure saved → {out_png}")
else:
    print("Not enough converged data points to plot.")

## 结论

In [ ]:
n_total = len(df)
n_conv  = df["converged"].sum()
print(f"Converged: {n_conv}/{n_total} trials")
if len(agg) >= 2:
    sl_i, _, r_i, *_ = stats.linregress(agg["d"], agg["mean_iters"])
    sl_t, _, r_t, *_ = stats.linregress(agg["d"], agg["mean_time"])
    print(f"  iters ~ {sl_i:+.1f} per d-unit   R²={r_i**2:.3f}")
    print(f"  time  ~ {sl_t:+.3f}s per d-unit  R²={r_t**2:.3f}")
    print(f"  CONV_TOL={CONV_TOL}  CONV_PATIENCE={CONV_PATIENCE}  T_MAX={T_MAX:,d}")